# Explaratory Data

## 0. Import Dataset

In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')
 
# Sastrawi untuk NLP Bahasa Indonesia
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# ─── Inisialisasi Sastrawi (sama persis dengan kode kamu) ────────────────────
stemmer  = StemmerFactory().create_stemmer()
stopword = StopWordRemoverFactory().create_stop_word_remover()

print("=" * 65)
print("  ANALISIS BAB 4 - SKRIPSI STACKING ENSEMBLE LEARNING")
print("  IDNHoaxCorpus - Universitas Pasundan")
print("=" * 65)

  ANALISIS BAB 4 - SKRIPSI STACKING ENSEMBLE LEARNING
  IDNHoaxCorpus - Universitas Pasundan


## 1. Load Dataset

In [39]:
print("\n[LOAD] Membaca dataset IDNHoaxCorpus...")
 
# Coba beberapa nama file yang mungkin dipakai
nama_file ='../data/raw/datasetUMPOHoax.csv'
df = pd.read_csv(nama_file)
       
if df is None:
    print("[ERROR] File dataset tidak ditemukan!")
    print("Pastikan file CSV ada di folder yang sama dengan script ini.")
    print(f"Nama file yang dicari: {nama_file}")
    exit()
 
# Deteksi kolom tweet dan label
kolom_teks  = 'tweet'  if 'tweet'  in df.columns else df.columns[2]
kolom_label = 'label'  if 'label'  in df.columns else df.columns[5]

sampel = df[kolom_teks].head(500).fillna('').astype(str)
 
print(f"[INFO] Kolom teks  : '{kolom_teks}'")
print(f"[INFO] Kolom label : '{kolom_label}'")
print(f"[INFO] Total data  : {len(df):,} baris")


[LOAD] Membaca dataset IDNHoaxCorpus...
[INFO] Kolom teks  : 'tweet'
[INFO] Kolom label : 'label'
[INFO] Total data  : 3,812 baris


## 2. Explaratory data EDA

### 2.1 Statistik Deskriptif Dataset

In [ ]:
print("\n[LOAD] Membaca dataset IDNHoaxCorpus...")

# ============================================================
# Mencoba membaca file dataset CSV
# ============================================================

# Nama file dataset yang akan dibaca
nama_file ='../data/raw/datasetUMPOHoax.csv'

# Membaca file CSV menggunakan pandas
df = pd.read_csv(nama_file)

# ============================================================
# Validasi apakah dataset berhasil dibaca
# ============================================================

# Jika dataframe kosong / tidak ditemukan
if df is None:
    print("[ERROR] File dataset tidak ditemukan!")
    print("Pastikan file CSV ada di folder yang sama dengan script ini.")
    print(f"Nama file yang dicari: {nama_file}")
    exit()

# ============================================================
# Menentukan kolom teks dan label secara otomatis
# ============================================================

# Jika ada kolom bernama 'tweet', gunakan sebagai kolom teks
# Jika tidak ada, gunakan kolom index ke-2
kolom_teks = 'tweet' if 'tweet' in df.columns else df.columns[2]

# Jika ada kolom bernama 'label', gunakan sebagai kolom label
# Jika tidak ada, gunakan kolom index ke-5
kolom_label = 'label' if 'label' in df.columns else df.columns[5]

# ============================================================
# Mengambil 500 data pertama sebagai sampel
# ============================================================

# Mengambil isi kolom teks sebanyak 500 baris pertama
# fillna('')      -> mengganti nilai kosong menjadi string kosong
# astype(str)     -> memastikan semua data bertipe string
sampel = df[kolom_teks].head(500).fillna('').astype(str)

# ============================================================
# Menampilkan informasi dataset
# ============================================================

# Menampilkan nama kolom teks yang digunakan
print(f"[INFO] Kolom teks  : '{kolom_teks}'")

# Menampilkan nama kolom label yang digunakan
print(f"[INFO] Kolom label : '{kolom_label}'")

# Menampilkan jumlah total data pada dataset
print(f"[INFO] Total data  : {len(df):,} baris")


[EDA 1A] Statistik Deskriptif Dataset...

Dimensi dataset : 3812 baris x 6 kolom
Kolom yang ada  : ['topik', 'keyword', 'tweet', 'gambar', 'url', 'label']

--- Statistik Deskriptif Jumlah Kata per Teks ---
Metrik                  Hoaks (Kelas 1)    Valid (Kelas 0)
----------------------------------------------------------
count                           3042.00             770.00
mean                              18.67              19.80
std                               10.44              11.07
min                                1.00               1.00
25%                               11.00              10.25
50%                               16.00              17.00
75%                               23.00              29.00
max                              123.00              49.00


### 2.2 Distribusi Kelas

In [ ]:
print("\n[EDA 1B] Distribusi Kelas (Label)...")

# ============================================================
# Menghitung jumlah data pada masing-masing kelas
# ============================================================

# value_counts() digunakan untuk menghitung frekuensi tiap label
distribusi = df[kolom_label].value_counts()

# ============================================================
# Menampilkan jumlah data tiap kelas
# ============================================================

# get(1,0) -> mengambil jumlah label 1 (Hoaks)
# get(0,0) -> mengambil jumlah label 0 (Valid)
print(f"\nKelas Hoaks  (1) : {distribusi.get(1, 0):,} data")
print(f"Kelas Valid  (0) : {distribusi.get(0, 0):,} data")

# ============================================================
# Menghitung rasio imbalance dataset
# ============================================================

# Rasio imbalance menunjukkan perbandingan jumlah data hoaks dan valid
# Semakin jauh dari 1:1 berarti dataset semakin tidak seimbang
print(f"Rasio Imbalance  : {distribusi.get(1,0)/distribusi.get(0,1):.2f}:1")

# ============================================================
# Membuat area visualisasi (2 subplot)
# ============================================================

# subplot 1 -> Bar Chart
# subplot 2 -> Pie Chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Memberikan judul utama visualisasi
fig.suptitle(
    'Distribusi Kelas Dataset IDNHoaxCorpus',
    fontsize=13,
    fontweight='bold',
    y=1.02
)

# ============================================================
# BAR CHART DISTRIBUSI KELAS
# ============================================================

# Mapping label angka menjadi nama kelas
label_names = {
    1: 'Hoaks (1)',
    0: 'Valid (0)'
}

# Warna untuk masing-masing kelas
colors = ['#E74C3C', '#2ECC71']

# Membuat diagram batang
bars = axes[0].bar(
    [label_names[k] for k in distribusi.index],  # nama label
    distribusi.values,                           # jumlah data
    color=colors,
    edgecolor='black',
    width=0.5
)

# Judul subplot pertama
axes[0].set_title('Jumlah Data per Kelas', fontweight='bold')

# Label sumbu Y
axes[0].set_ylabel('Jumlah Dokumen')

# ============================================================
# Menambahkan angka di atas batang diagram
# ============================================================

for bar, val in zip(bars, distribusi.values):

    # Menampilkan jumlah data di atas masing-masing batang
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 30,
        f'{val:,}',
        ha='center',
        va='bottom',
        fontweight='bold'
    )

# Mengatur batas maksimum sumbu Y
axes[0].set_ylim(0, max(distribusi.values) * 1.15)

# ============================================================
# PIE CHART PROPORSI KELAS
# ============================================================

axes[1].pie(
    distribusi.values,

    # Label tiap potongan pie chart
    labels=[label_names[k] for k in distribusi.index],

    # Menampilkan persentase
    autopct='%1.1f%%',

    # Warna pie chart
    colors=colors,

    # Rotasi awal pie chart
    startangle=90,

    # Garis tepi antar potongan
    wedgeprops={
        'edgecolor': 'white',
        'linewidth': 2
    }
)

# Judul subplot kedua
axes[1].set_title('Proporsi Kelas (%)', fontweight='bold')

# ============================================================
# Mengatur layout agar tidak bertabrakan
# ============================================================

plt.tight_layout()

# ============================================================
# Menyimpan hasil visualisasi
# ============================================================

plt.savefig(
    '../gambar/gambar_4X_distribusi_kelas.png',
    dpi=150,
    bbox_inches='tight'
)

# Menutup figure agar memori lebih hemat
plt.close()

# ============================================================
# Informasi lokasi penyimpanan gambar
# ============================================================

print("[SAVE] output_bab4/gambar_4X_distribusi_kelas.png")


[EDA 1B] Distribusi Kelas (Label)...

Kelas Hoaks  (1) : 3,042 data
Kelas Valid  (0) : 770 data
Rasio Imbalance  : 3.95:1
[SAVE] output_bab4/gambar_4X_distribusi_kelas.png


### 2.3 Distribusi Panjang Teks Per Kelas

In [ ]:
print("\n[EDA 1C] Distribusi Panjang Teks per Kelas...")

# ============================================================
# VISUALISASI DISTRIBUSI PANJANG TEKS
# ============================================================

# Analisis ini bertujuan untuk melihat:
# 1. Apakah teks hoaks cenderung lebih panjang atau pendek
# 2. Apakah distribusi jumlah kata antar kelas berbeda
# 3. Apakah terdapat outlier pada panjang teks
# 4. Karakteristik umum dataset berdasarkan jumlah kata

# ============================================================
# MEMBUAT AREA VISUALISASI
# ============================================================

# Membuat figure dengan 2 subplot
# subplot kiri  -> Histogram distribusi jumlah kata
# subplot kanan -> Boxplot perbandingan distribusi

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Menambahkan judul utama visualisasi
fig.suptitle(
    'Distribusi Panjang Teks per Kelas',
    fontsize=13,
    fontweight='bold'
)

# ============================================================
# HISTOGRAM DISTRIBUSI JUMLAH KATA
# ============================================================

# Looping untuk masing-masing kelas:
# label = 1 -> Hoaks
# label = 0 -> Valid

for label, color, nama in [

    # Kelas hoaks menggunakan warna merah
    (1, '#E74C3C', 'Hoaks'),

    # Kelas valid menggunakan warna hijau
    (0, '#2ECC71', 'Valid')

]:

    # ========================================================
    # Mengambil subset data berdasarkan kelas
    # ========================================================

    # Contoh:
    # Jika label = 1 maka hanya mengambil data hoaks
    # Jika label = 0 maka hanya mengambil data valid

    data_subset = df[df[kolom_label] == label]['jumlah_kata']

    # ========================================================
    # Membuat histogram distribusi jumlah kata
    # ========================================================

    axes[0].hist(

        # Data jumlah kata
        data_subset,

        # Jumlah interval histogram
        # Semakin besar bins maka grafik semakin detail
        bins=40,

        # Transparansi warna
        alpha=0.7,

        # Warna histogram
        color=color,

        # Nama label pada legend
        label=nama,

        # Menghilangkan garis tepi batang histogram
        edgecolor='none'
    )

# ============================================================
# PENGATURAN HISTOGRAM
# ============================================================

# Judul subplot histogram
axes[0].set_title(
    'Distribusi Jumlah Kata per Teks',
    fontweight='bold'
)

# Label sumbu X
axes[0].set_xlabel('Jumlah Kata')

# Label sumbu Y
axes[0].set_ylabel('Frekuensi')

# Menampilkan legenda
axes[0].legend()

# ============================================================
# MENAMBAHKAN GARIS MEDIAN
# ============================================================

# Median digunakan untuk melihat titik tengah distribusi data
# Median lebih stabil dibanding rata-rata jika ada outlier

axes[0].axvline(

    # Posisi garis median
    df['jumlah_kata'].median(),

    # Warna garis
    color='navy',

    # Jenis garis putus-putus
    linestyle='--',

    # Label pada legenda
    label=f"Median: {df['jumlah_kata'].median():.0f}"
)

# ============================================================
# BOXPLOT PERBANDINGAN DISTRIBUSI
# ============================================================

# Membuat dataframe baru khusus visualisasi
# agar data asli tidak berubah

df_plot = df[[kolom_label, 'jumlah_kata']].copy()

# ============================================================
# Mengubah label numerik menjadi teks
# ============================================================

# 1 -> Hoaks
# 0 -> Valid

df_plot['Kelas'] = df_plot[kolom_label].map({
    1: 'Hoaks',
    0: 'Valid'
})

# ============================================================
# MEMBUAT BOXPLOT
# ============================================================

# Boxplot digunakan untuk melihat:
# 1. Median
# 2. Persebaran data
# 3. Quartile
# 4. Outlier
# 5. Perbedaan distribusi antar kelas

sns.boxplot(

    # Dataset yang digunakan
    data=df_plot,

    # Sumbu X = kelas
    x='Kelas',

    # Sumbu Y = jumlah kata
    y='jumlah_kata',

    # Warna tiap kelas
    palette={
        'Hoaks': '#E74C3C',
        'Valid': '#2ECC71'
    },

    # Menentukan subplot tujuan
    ax=axes[1]
)

# ============================================================
# PENGATURAN BOXPLOT
# ============================================================

# Judul subplot boxplot
axes[1].set_title(
    'Perbandingan Distribusi Jumlah Kata',
    fontweight='bold'
)

# Label sumbu X
axes[1].set_xlabel('Kelas')

# Label sumbu Y
axes[1].set_ylabel('Jumlah Kata')

# ============================================================
# MENGATUR SPASI LAYOUT
# ============================================================

# Agar antar grafik tidak saling bertabrakan
plt.tight_layout()

# ============================================================
# MENYIMPAN HASIL VISUALISASI
# ============================================================

# Menyimpan gambar ke folder gambar
# dpi=150 -> kualitas gambar lebih tajam
# bbox_inches='tight' -> menghindari bagian gambar terpotong

plt.savefig(
    '../gambar/gambar_4X_distribusi_panjang_teks.png',
    dpi=150,
    bbox_inches='tight'
)

# ============================================================
# MENUTUP FIGURE
# ============================================================

# plt.close() digunakan agar memori lebih hemat
# terutama jika banyak visualisasi dibuat secara berulang

plt.close()

# ============================================================
# MENAMPILKAN INFORMASI PENYIMPANAN
# ============================================================

print("[SAVE] output_bab4/gambar_4X_distribusi_panjang_teks.png")


[EDA 1C] Distribusi Panjang Teks per Kelas...
[SAVE] output_bab4/gambar_4X_distribusi_panjang_teks.png


### 2.4 Top 20 Kata Paling Sering per Kelas (sebelum preprocessing)

In [ ]:
print("\n[EDA 1D] Top 20 Kata per Kelas (sebelum preprocessing)...")

# ============================================================
# IMPORT LIBRARY
# ============================================================

# Counter digunakan untuk menghitung frekuensi kemunculan kata
# secara otomatis dan efisien

from collections import Counter

# ============================================================
# FUNGSI UNTUK MENGHITUNG TOP KATA
# ============================================================

def hitung_top_kata(df_subset, kolom, n=20):

    # ========================================================
    # MENGGABUNGKAN SELURUH TEKS
    # ========================================================

    # astype(str)
    # memastikan seluruh isi menjadi string

    # str.lower()
    # mengubah seluruh huruf menjadi lowercase
    # agar "Presiden" dan "presiden" dianggap sama

    # ' '.join(...)
    # menggabungkan semua teks menjadi satu string panjang

    # split()
    # memisahkan teks menjadi token/kata

    semua_kata = ' '.join(
        df_subset[kolom]
        .astype(str)
        .str.lower()
    ).split()

    # ========================================================
    # STOPWORDS SEDERHANA
    # ========================================================

    # Daftar kata umum yang tidak memiliki makna penting
    # dalam analisis teks

    # Kata-kata seperti:
    # "yang", "dan", "di", dll
    # biasanya muncul sangat sering tetapi
    # tidak membantu membedakan kelas

    stopwords_kasar = {

        'yang', 'di', 'dan', 'ke', 'dari',
        'ini', 'itu', 'dengan', 'untuk',
        'pada', 'adalah', 'akan', 'atau',
        'juga', 'tidak', 'ada', 'dalam',
        'oleh', 'kita', 'kami', 'mereka',
        'saya', 'kamu', 'dia', 'nya'
    }

    # ========================================================
    # FILTERING KATA
    # ========================================================

    # Proses filtering dilakukan untuk:
    # 1. Menghapus stopwords
    # 2. Menghapus kata terlalu pendek
    #    (misalnya: "ya", "ok", "ga")

    kata_bersih = [

        # Ambil kata
        k

        # Iterasi semua kata
        for k in semua_kata

        # Jika kata bukan stopword
        if k not in stopwords_kasar

        # Dan panjang kata > 2 karakter
        and len(k) > 2
    ]

    # ========================================================
    # MENGHITUNG FREKUENSI KATA
    # ========================================================

    # Counter() menghitung jumlah kemunculan tiap kata

    # most_common(n)
    # mengambil n kata paling sering muncul

    return Counter(kata_bersih).most_common(n)

# ============================================================
# MENGAMBIL TOP KATA UNTUK MASING-MASING KELAS
# ============================================================

# ============================================================
# TOP KATA KELAS HOAKS
# ============================================================

# Mengambil data dengan label = 1
# lalu menghitung 20 kata paling sering

top_hoaks = hitung_top_kata(

    # Filter hanya data hoaks
    df[df[kolom_label] == 1],

    # Kolom teks
    kolom_teks
)

# ============================================================
# TOP KATA KELAS VALID
# ============================================================

# Mengambil data dengan label = 0
# lalu menghitung 20 kata paling sering

top_valid = hitung_top_kata(

    # Filter hanya data valid
    df[df[kolom_label] == 0],

    # Kolom teks
    kolom_teks
)

# ============================================================
# MEMBUAT AREA VISUALISASI
# ============================================================

# Membuat figure dengan 2 subplot horizontal
# kiri  -> kata kelas hoaks
# kanan -> kata kelas valid

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ============================================================
# JUDUL UTAMA VISUALISASI
# ============================================================

fig.suptitle(
    'Top 20 Kata Paling Sering per Kelas (Sebelum Preprocessing)',
    fontsize=12,
    fontweight='bold'
)

# ============================================================
# MEMBUAT VISUALISASI UNTUK MASING-MASING KELAS
# ============================================================

for ax, data, judul, warna in [

    # Visualisasi kelas hoaks
    (axes[0], top_hoaks, 'Kelas Hoaks', '#E74C3C'),

    # Visualisasi kelas valid
    (axes[1], top_valid, 'Kelas Valid', '#2ECC71')
]:

    # ========================================================
    # MEMISAHKAN KATA DAN FREKUENSI
    # ========================================================

    # Mengambil daftar kata saja
    kata_list = [k for k, _ in data]

    # Mengambil daftar frekuensi saja
    freq_list = [f for _, f in data]

    # ========================================================
    # MEMBUAT HORIZONTAL BAR CHART
    # ========================================================

    bars = ax.barh(

        # Posisi batang
        range(len(kata_list)),

        # Nilai frekuensi
        freq_list,

        # Warna batang
        color=warna,

        # Transparansi
        alpha=0.8,

        # Warna garis tepi
        edgecolor='white'
    )

    # ========================================================
    # MENGATUR LABEL Y
    # ========================================================

    # Posisi label kata
    ax.set_yticks(range(len(kata_list)))

    # Nama kata
    ax.set_yticklabels(kata_list, fontsize=10)

    # Membalik urutan agar frekuensi terbesar di atas
    ax.invert_yaxis()

    # ========================================================
    # PENGATURAN JUDUL DAN LABEL
    # ========================================================

    # Judul subplot
    ax.set_title(
        judul,
        fontweight='bold',
        fontsize=12
    )

    # Label sumbu X
    ax.set_xlabel('Frekuensi Kemunculan')

    # ========================================================
    # MENAMPILKAN ANGKA FREKUENSI DI SAMPING BATANG
    # ========================================================

    for bar, val in zip(bars, freq_list):

        # Menambahkan teks angka frekuensi
        ax.text(

            # Posisi X teks
            val + max(freq_list)*0.01,

            # Posisi Y teks
            bar.get_y() + bar.get_height()/2,

            # Isi teks
            str(val),

            # Posisi vertikal
            va='center',

            # Ukuran font
            fontsize=9
        )

# ============================================================
# MENGATUR LAYOUT
# ============================================================

# Agar antar grafik tidak bertabrakan
plt.tight_layout()

# ============================================================
# MENYIMPAN VISUALISASI
# ============================================================

# Menyimpan hasil grafik ke folder gambar

plt.savefig(

    '../gambar/gambar_4X_top20_kata_per_kelas.png',

    # Resolusi gambar
    dpi=150,

    # Menghindari gambar terpotong
    bbox_inches='tight'
)

# ============================================================
# MENUTUP FIGURE
# ============================================================

# Untuk menghemat memori
plt.close()

# ============================================================
# INFORMASI PENYIMPANAN FILE
# ============================================================

print("[SAVE] output_bab4/gambar_4X_top20_kata_per_kelas.png")


[EDA 1D] Top 20 Kata per Kelas (sebelum preprocessing)...
[SAVE] output_bab4/gambar_4X_top20_kata_per_kelas.png


## 3. Missing Values

In [ ]:
print("\n" + "=" * 65)

# ============================================================
# JUDUL BAGIAN ANALISIS
# ============================================================

# Menampilkan heading agar output notebook lebih rapi
# dan mudah dibedakan antar tahap preprocessing

print("  BAGIAN 2: PENGECEKAN MISSING VALUES")

# Informasi posisi tahapan dalam alur penelitian
# Tahapan ini dilakukan sebelum data cleansing

print("  Penempatan: Dalam 'Persiapan Data', SEBELUM 'Data Cleansing'")

print("=" * 65)

# ============================================================
# ANALISIS MISSING VALUES
# ============================================================

print("\n[MISSING] Mengecek nilai kosong per kolom...")

# ============================================================
# MEMBUAT TABEL INFORMASI MISSING VALUES
# ============================================================

# DataFrame ini digunakan untuk merangkum:
# 1. Nama kolom
# 2. Tipe data tiap kolom
# 3. Jumlah missing values
# 4. Persentase missing values

missing_info = pd.DataFrame({

    # Nama seluruh kolom dataset
    'Kolom': df.columns,

    # Tipe data masing-masing kolom
    # contoh: object, int64, float64
    'Tipe Data': df.dtypes.values,

    # Menghitung jumlah nilai kosong per kolom
    # isnull() -> mendeteksi nilai kosong
    # sum()    -> menghitung total True
    'Jumlah Missing': df.isnull().sum().values,

    # Menghitung persentase missing value
    # dibagi total data lalu dikali 100
    'Persentase (%)': (

        df.isnull().sum().values / len(df) * 100

    ).round(2)  # dibulatkan 2 angka desimal
})

# ============================================================
# MENAMPILKAN TABEL MISSING VALUES
# ============================================================

# to_string(index=False)
# digunakan agar tabel tampil rapi tanpa index

print(missing_info.to_string(index=False))

# ============================================================
# MENGHITUNG TOTAL KESELURUHAN MISSING VALUES
# ============================================================

# sum().sum()
# sum pertama -> jumlah missing tiap kolom
# sum kedua   -> total seluruh missing dataset

total_missing = df.isnull().sum().sum()

# Menampilkan total missing values
print(f"\nTotal keseluruhan nilai kosong : {total_missing}")

# ============================================================
# PENGAMBILAN KEPUTUSAN BERDASARKAN HASIL ANALISIS
# ============================================================

# ============================================================
# JIKA TIDAK ADA MISSING VALUE
# ============================================================

if total_missing == 0:

    print("[HASIL] Dataset BERSIH - tidak ada missing values sama sekali.")

    print("[KEPUTUSAN] Tidak diperlukan penanganan missing values.")

# ============================================================
# JIKA ADA MISSING VALUE
# ============================================================

else:

    # Menampilkan jumlah missing values yang ditemukan
    print(f"[HASIL] Ditemukan {total_missing} missing values.")

    # Menjelaskan keputusan preprocessing
    print("[KEPUTUSAN] Baris yang mengandung missing values pada kolom")

    print("            'tweet' dan 'label' akan dihapus (drop).")

    # ========================================================
    # MENYIMPAN JUMLAH DATA SEBELUM PENGHAPUSAN
    # ========================================================

    df_before = len(df)

    # ========================================================
    # MENGHAPUS BARIS YANG MEMILIKI NILAI KOSONG
    # ========================================================

    # dropna(subset=[...])
    # hanya mengecek kolom tertentu

    df = df.dropna(subset=[kolom_teks, kolom_label])

    # ========================================================
    # MENAMPILKAN PERUBAHAN JUMLAH DATA
    # ========================================================

    print(
        f"[INFO] Data berkurang dari {df_before:,} "
        f"menjadi {len(df):,} baris."
    )

# ============================================================
# VISUALISASI MISSING VALUES
# ============================================================

# Visualisasi bertujuan untuk:
# 1. Melihat kolom mana yang memiliki missing value
# 2. Membandingkan jumlah missing antar kolom
# 3. Memberikan bukti visual pada laporan penelitian

fig, ax = plt.subplots(figsize=(10, 5))

# ============================================================
# MENGHITUNG MISSING VALUE PER KOLOM
# ============================================================

missing_per_col = df.isnull().sum()

# ============================================================
# MENENTUKAN WARNA BATANG
# ============================================================

# Hijau  -> tidak ada missing value
# Merah  -> ada missing value

colors_mv = [

    '#E74C3C' if v > 0 else '#2ECC71'

    for v in missing_per_col.values
]

# ============================================================
# MEMBUAT BAR CHART
# ============================================================

bars = ax.bar(

    # Nama kolom
    missing_per_col.index,

    # Jumlah missing values
    missing_per_col.values,

    # Warna batang
    color=colors_mv,

    # Warna garis tepi
    edgecolor='black',

    # Transparansi warna
    alpha=0.85
)

# ============================================================
# PENGATURAN JUDUL DAN LABEL
# ============================================================

# Judul grafik
ax.set_title(
    'Jumlah Missing Values per Kolom Dataset',
    fontweight='bold',
    pad=15
)

# Label sumbu X
ax.set_xlabel('Nama Kolom')

# Label sumbu Y
ax.set_ylabel('Jumlah Missing Values')

# ============================================================
# MENAMPILKAN ANGKA DI ATAS BATANG
# ============================================================

for bar, val in zip(bars, missing_per_col.values):

    ax.text(

        # Posisi horizontal teks
        bar.get_x() + bar.get_width()/2,

        # Posisi vertikal teks
        bar.get_height() +
        max(missing_per_col.values) * 0.02 + 0.5,

        # Isi teks
        str(val),

        # Posisi horizontal teks
        ha='center',

        # Posisi vertikal teks
        va='bottom',

        # Ketebalan font
        fontweight='bold',

        # Ukuran font
        fontsize=11
    )

# ============================================================
# MENGATUR BATAS MAKSIMUM SUMBU Y
# ============================================================

ax.set_ylim(

    0,

    # Memberi ruang ekstra di atas batang
    max(missing_per_col.values) * 1.25 + 10
)

# ============================================================
# MEMBUAT LEGENDA WARNA
# ============================================================

from matplotlib.patches import Patch

legend = [

    # Warna hijau
    Patch(
        color='#2ECC71',
        label='Tidak ada missing value'
    ),

    # Warna merah
    Patch(
        color='#E74C3C',
        label='Ada missing value'
    )
]

# Menampilkan legenda
ax.legend(handles=legend, loc='upper right')

# ============================================================
# MEMUTAR LABEL KOLOM
# ============================================================

# Agar nama kolom panjang tidak bertabrakan

plt.xticks(rotation=30, ha='right')

# ============================================================
# MENGATUR TATA LETAK
# ============================================================

plt.tight_layout()

# ============================================================
# MENYIMPAN HASIL VISUALISASI
# ============================================================

plt.savefig(

    '../gambar/gambar_4X_missing_values.png',

    # Resolusi gambar
    dpi=150,

    # Menghindari gambar terpotong
    bbox_inches='tight'
)

# ============================================================
# MENUTUP FIGURE
# ============================================================

# Menghemat memori setelah grafik selesai dibuat
plt.close()

# ============================================================
# INFORMASI PENYIMPANAN FILE
# ============================================================

print("[SAVE] output_bab4/gambar_4X_missing_values.png")


  BAGIAN 2: PENGECEKAN MISSING VALUES
  Penempatan: Dalam 'Persiapan Data', SEBELUM 'Data Cleansing'

[MISSING] Mengecek nilai kosong per kolom...
       Kolom Tipe Data  Jumlah Missing  Persentase (%)
       topik    object             970           25.55
     keyword    object             947           24.94
       tweet    object               0            0.00
      gambar    object            3734           98.34
         url    object              46            1.21
       label     int64               0            0.00
panjang_teks     int64               0            0.00
 jumlah_kata     int64               0            0.00

Total keseluruhan nilai kosong : 5697
[HASIL] Ditemukan 5697 missing values.
[KEPUTUSAN] Baris yang mengandung missing values pada kolom
            'tweet' dan 'label' akan dihapus (drop).
[INFO] Data berkurang dari 3,797 menjadi 3,797 baris.
[SAVE] output_bab4/gambar_4X_missing_values.png


## 4. Data Duplikat

In [ ]:
print("\n" + "=" * 65)

# ============================================================
# JUDUL BAGIAN ANALISIS
# ============================================================

# Menampilkan heading tahapan preprocessing
# agar output notebook lebih terstruktur

print("  BAGIAN 3: PENGECEKAN DATA DUPLIKAT")

# Menjelaskan posisi tahapan dalam alur penelitian
# dilakukan setelah pengecekan missing value

print("  Penempatan: Setelah 'Missing Values', sebelum 'Data Cleansing'")

print("=" * 65)

# ============================================================
# PENGECEKAN DATA DUPLIKAT
# ============================================================

print("\n[DUPLIKAT] Mengecek baris duplikat...")

# ============================================================
# MENGHITUNG DUPLIKAT BERDASARKAN KOLOM TEKS
# ============================================================

# duplicated(subset=[kolom_teks])
# digunakan untuk mengecek apakah ada isi tweet
# yang sama persis dengan tweet lainnya

# sum()
# menghitung total nilai True

jumlah_duplikat = df.duplicated(
    subset=[kolom_teks]
).sum()

# ============================================================
# MENGHITUNG DUPLIKAT BERDASARKAN SEMUA KOLOM
# ============================================================

# duplicated() tanpa subset
# mengecek apakah seluruh isi baris sama persis

jumlah_duplikat_full = df.duplicated().sum()

# ============================================================
# MENAMPILKAN HASIL ANALISIS DUPLIKAT
# ============================================================

print(
    f"Duplikat berdasarkan kolom 'tweet' saja : "
    f"{jumlah_duplikat:,} baris"
)

print(
    f"Duplikat berdasarkan semua kolom        : "
    f"{jumlah_duplikat_full:,} baris"
)

# ============================================================
# JIKA DITEMUKAN DATA DUPLIKAT
# ============================================================

if jumlah_duplikat > 0:

    # ========================================================
    # MENAMPILKAN CONTOH DATA DUPLIKAT
    # ========================================================

    print("\n[CONTOH] 3 Contoh baris duplikat yang ditemukan:")

    # duplicated(..., keep=False)
    # digunakan agar seluruh pasangan duplikat tampil

    # head(6)
    # mengambil 6 baris pertama
    # biasanya mewakili 3 pasangan data duplikat

    duplikat_contoh = df[
        df.duplicated(
            subset=[kolom_teks],
            keep=False
        )
    ].head(6)

    # Menampilkan kolom teks dan label saja
    print(
        duplikat_contoh[
            [kolom_teks, kolom_label]
        ].to_string()
    )

    # ========================================================
    # MENYIMPAN JUMLAH DATA SEBELUM PENGHAPUSAN
    # ========================================================

    df_before = len(df)

    # ========================================================
    # MENGHAPUS DATA DUPLIKAT
    # ========================================================

    # drop_duplicates(subset=[kolom_teks])
    # hanya mempertahankan satu tweet unik

    # Tujuan:
    # 1. Mengurangi bias model
    # 2. Menghindari data berulang
    # 3. Meningkatkan kualitas dataset

    df = df.drop_duplicates(
        subset=[kolom_teks]
    )

    # ========================================================
    # RESET INDEX DATAFRAME
    # ========================================================

    # reset_index(drop=True)
    # digunakan agar index kembali rapi
    # setelah beberapa baris dihapus

    df = df.reset_index(drop=True)

    # ========================================================
    # MENAMPILKAN HASIL PENGHAPUSAN
    # ========================================================

    print(
        f"\n[KEPUTUSAN] {jumlah_duplikat} "
        f"baris duplikat DIHAPUS."
    )

    print(
        f"[INFO] Data berkurang dari "
        f"{df_before:,} menjadi {len(df):,} baris."
    )

# ============================================================
# JIKA TIDAK ADA DUPLIKAT
# ============================================================

else:

    print("\n[HASIL] Dataset BERSIH - tidak ada data duplikat.")

    print("[KEPUTUSAN] Tidak diperlukan penghapusan duplikat.")

# ============================================================
# VISUALISASI JUMLAH DATA
# ============================================================

# Visualisasi ini bertujuan untuk:
# 1. Membandingkan jumlah data tiap tahap
# 2. Menunjukkan efek preprocessing
# 3. Memberikan bukti visual pada laporan penelitian

fig, ax = plt.subplots(figsize=(8, 5))

# ============================================================
# LABEL KATEGORI TAHAP PEMROSESAN
# ============================================================

kategori = [

    # Jumlah data awal dataset
    'Data Asli',

    # Setelah penghapusan missing values
    'Setelah Hapus\nMissing Values',

    # Setelah penghapusan duplikat
    'Setelah Hapus\nDuplikat'
]

# ============================================================
# NILAI JUMLAH DATA
# ============================================================

nilai = [

    # Jumlah data asli
    3812,

    # Jumlah setelah missing value
    len(df) + jumlah_duplikat,

    # Jumlah akhir setelah hapus duplikat
    len(df)
]

# ============================================================
# WARNA MASING-MASING BATANG
# ============================================================

colors_d = [

    # Biru -> data awal
    '#3498DB',

    # Oranye -> setelah missing value
    '#F39C12',

    # Hijau -> setelah duplikat
    '#2ECC71'
]

# ============================================================
# MEMBUAT BAR CHART
# ============================================================

bars = ax.bar(

    # Label kategori
    kategori,

    # Nilai jumlah data
    nilai,

    # Warna batang
    color=colors_d,

    # Warna garis tepi
    edgecolor='black',

    # Transparansi
    alpha=0.85,

    # Lebar batang
    width=0.5
)

# ============================================================
# PENGATURAN JUDUL DAN LABEL
# ============================================================

# Judul grafik
ax.set_title(

    'Jumlah Data pada Setiap Tahap Pembersihan Awal',

    fontweight='bold',

    pad=15
)

# Label sumbu Y
ax.set_ylabel('Jumlah Baris Data')

# ============================================================
# MENAMPILKAN ANGKA DI ATAS BATANG
# ============================================================

for bar, val in zip(bars, nilai):

    ax.text(

        # Posisi horizontal teks
        bar.get_x() + bar.get_width()/2,

        # Posisi vertikal teks
        bar.get_height() + 30,

        # Isi teks
        f'{val:,}',

        # Posisi horizontal teks
        ha='center',

        # Posisi vertikal teks
        va='bottom',

        # Ketebalan font
        fontweight='bold',

        # Ukuran font
        fontsize=12
    )

# ============================================================
# MENGATUR BATAS SUMBU Y
# ============================================================

# Memberikan ruang di atas batang
# agar teks tidak terpotong

ax.set_ylim(

    0,

    max(nilai) * 1.15
)

# ============================================================
# MENGATUR TATA LETAK
# ============================================================

plt.tight_layout()

# ============================================================
# MENYIMPAN VISUALISASI
# ============================================================

plt.savefig(

    '../gambar/gambar_4X_jumlah_data_pembersihan_awal.png',

    # Resolusi gambar
    dpi=150,

    # Menghindari gambar terpotong
    bbox_inches='tight'
)

# ============================================================
# MENUTUP FIGURE
# ============================================================

# Untuk menghemat penggunaan memori
plt.close()

# ============================================================
# INFORMASI PENYIMPANAN FILE
# ============================================================

print("[SAVE] output_bab4/gambar_4X_jumlah_data_pembersihan_awal.png")


  BAGIAN 3: PENGECEKAN DATA DUPLIKAT
  Penempatan: Setelah 'Missing Values', sebelum 'Data Cleansing'

[DUPLIKAT] Mengecek baris duplikat...
Duplikat berdasarkan kolom 'tweet' saja : 0 baris
Duplikat berdasarkan semua kolom        : 0 baris

[HASIL] Dataset BERSIH - tidak ada data duplikat.
[KEPUTUSAN] Tidak diperlukan penghapusan duplikat.
[SAVE] output_bab4/gambar_4X_jumlah_data_pembersihan_awal.png


## 5. DAMPAK KUANTITATIF TIAP TAHAP PREPROCESSING

### 5.1 Fungsi minimal dan maksimal cleaning

In [25]:
def bersihkan_teks_total(teks):
    teks = str(teks).lower()
    teks = re.sub(r'http\S+|www\S+|https\S+', '', teks, flags=re.MULTILINE)
    teks = re.sub(r'\@\w+|\#', '', teks)
    teks = re.sub(r'[^a-zA-Z\s]', '', teks)
    teks = stopword.remove(teks)
    teks = stemmer.stem(teks)
    return teks
 
def bersihkan_teks_minimal(teks):
    teks = str(teks).lower()
    teks = re.sub(r'http\S+|www\S+|https\S+', '', teks, flags=re.MULTILINE)
    return teks

In [26]:
def tahap_lowercase(teks):
    return str(teks).lower()
 
def tahap_cleaning(teks):
    # Semua cleaning sebelum stopword (dari bersihkan_teks_total)
    teks = re.sub(r'http\S+|www\S+|https\S+', '', teks, flags=re.MULTILINE)
    teks = re.sub(r'\@\w+|\#', '', teks)
    teks = re.sub(r'[^a-zA-Z\s]', '', teks)
    teks = re.sub(r'\s+', ' ', teks).strip()
    return teks
 
def tahap_hapus_url(teks):
    # Khusus pipeline minimal — hanya hapus URL
    return re.sub(r'http\S+|www\S+|https\S+', '', teks, flags=re.MULTILINE)
 
def tahap_stopword(teks):
    return stopword.remove(teks)
 
def tahap_stemming(teks):
    return stemmer.stem(teks)
 
def potong(teks, n=90):
    return teks[:n] + ('...' if len(teks) > n else '')

### 5.2 HITUNG VOCABULARY PER TAHAP — PIPELINE MAKSIMAL Raw → Lowercase → Cleaning → Stopword → Stemming

In [ ]:
print("\n" + "=" * 65)

# ============================================================
# JUDUL ANALISIS VOCABULARY
# ============================================================

# Analisis ini bertujuan untuk melihat:
# 1. Jumlah kata unik pada setiap tahap preprocessing
# 2. Efektivitas preprocessing dalam mengurangi noise
# 3. Dampak cleaning, stopword, dan stemming
# 4. Perubahan kompleksitas vocabulary dataset

print("  MENGHITUNG VOCABULARY — PIPELINE MAKSIMAL")

print("=" * 65)

# ============================================================
# DICTIONARY UNTUK MENYIMPAN HASIL
# ============================================================

# vocab_maks akan menyimpan:
# key   -> nama tahapan preprocessing
# value -> jumlah vocabulary (kata unik)

vocab_maks = {}

# ============================================================
# TAHAP 1 — RAW TEXT (DATA ASLI)
# ============================================================

# s1 = sampel data asli sebelum preprocessing

s1 = sampel

# ============================================================
# MENGHITUNG JUMLAH KATA UNIK
# ============================================================

# ' '.join(s1)
# menggabungkan seluruh teks menjadi satu string besar

# split()
# memecah teks menjadi daftar kata/token

# set()
# mengambil hanya kata unik

# len()
# menghitung jumlah kata unik

vocab_maks['1. Raw\n(Asli)'] = len(
    set(
        ' '.join(s1).split()
    )
)

# ============================================================
# TAHAP 2 — LOWERCASE
# ============================================================

# Mengubah seluruh huruf menjadi kecil
# contoh:
# "Presiden" -> "presiden"
# "COVID"    -> "covid"

# Tujuan:
# agar kata dengan makna sama tidak dianggap berbeda

s2 = s1.apply(tahap_lowercase)

# Menghitung vocabulary setelah lowercase
vocab_maks['2. Lowercase'] = len(
    set(
        ' '.join(s2).split()
    )
)

# ============================================================
# TAHAP 3 — CLEANING
# ============================================================

print("   [Cleaning...]")

# ============================================================
# PROSES CLEANING
# ============================================================

# tahap_cleaning biasanya berisi:
# 1. Menghapus URL
# 2. Menghapus mention (@user)
# 3. Menghapus hashtag
# 4. Menghapus angka
# 5. Menghapus simbol
# 6. Menghapus karakter non-huruf

# Tujuan:
# mengurangi noise pada data teks

s3 = s2.apply(tahap_cleaning)

# Menghitung vocabulary setelah cleaning
vocab_maks['3. Cleaning\n(URL+mention\n+hashtag\n+non-huruf)'] = len(
    set(
        ' '.join(s3).split()
    )
)

# ============================================================
# TAHAP 4 — STOPWORD REMOVAL
# ============================================================

print("   [Stopword Removal...]")

# ============================================================
# PROSES STOPWORD REMOVAL
# ============================================================

# Stopword adalah kata umum yang sering muncul
# tetapi kurang memiliki makna penting

# Contoh:
# "dan", "yang", "di", "ke", dll

# Tujuan:
# mengurangi kata yang tidak informatif

s4 = s3.apply(tahap_stopword)

# Menghitung vocabulary setelah stopword removal
vocab_maks['4. Stopword\nRemoval'] = len(
    set(
        ' '.join(s4).split()
    )
)

# ============================================================
# TAHAP 5 — STEMMING
# ============================================================

print("   [Stemming... harap tunggu ~30-60 detik]")

# ============================================================
# PROSES STEMMING
# ============================================================

# Stemming mengubah kata menjadi bentuk dasar

# Contoh:
# "berlari"   -> "lari"
# "makanan"   -> "makan"
# "membeli"   -> "beli"

# Tujuan:
# menyatukan variasi kata agar vocabulary lebih kecil
# dan representasi fitur lebih konsisten

# Catatan:
# stemming bahasa Indonesia cukup berat
# sehingga proses dapat memakan waktu

s5 = s4.apply(tahap_stemming)

# Menghitung vocabulary akhir setelah stemming
vocab_maks['5. Stemming\n(Output Akhir)'] = len(
    set(
        ' '.join(s5).split()
    )
)

# ============================================================
# MENYIMPAN JUMLAH VOCABULARY AWAL
# ============================================================

# Vocabulary raw digunakan sebagai pembanding
# untuk menghitung persentase reduksi

vocab_raw = vocab_maks['1. Raw\n(Asli)']

# ============================================================
# MENAMPILKAN HEADER TABEL
# ============================================================

# <38  -> rata kiri lebar 38 karakter
# >10  -> rata kanan lebar 10 karakter

print(f"\n{'Tahap':<38} {'Kata Unik':>10} {'Reduksi':>10}")

# Garis pemisah tabel
print("-" * 60)

# ============================================================
# MENAMPILKAN HASIL TIAP TAHAP
# ============================================================

for t, v in vocab_maks.items():

    # ========================================================
    # MENGHITUNG PERSENTASE REDUKSI VOCABULARY
    # ========================================================

    # Rumus:
    # (1 - vocabulary_sekarang / vocabulary_awal) * 100

    # Semakin besar reduksi:
    # berarti preprocessing semakin berhasil
    # mengurangi variasi kata/noise

    print(

        # Nama tahap preprocessing
        f"{t.replace(chr(10),' '):<38}"

        # Jumlah vocabulary
        f"{v:>10,}"

        # Persentase reduksi
        f"{(1-v/vocab_raw)*100:>9.1f}%"
    )


  MENGHITUNG VOCABULARY — PIPELINE MAKSIMAL
   [Cleaning...]
   [Stopword Removal...]
   [Stemming... harap tunggu ~30-60 detik]

Tahap                                   Kata Unik    Reduksi
------------------------------------------------------------
1. Raw (Asli)                               5,080       0.0%
2. Lowercase                                4,383      13.7%
3. Cleaning (URL+mention +hashtag +non-huruf)      3,430      32.5%
4. Stopword Removal                         3,375      33.6%
5. Stemming (Output Akhir)                  2,853      43.8%


### 5.3 HITUNG VOCABULARY PER TAHAP — PIPELINE MINIMAL Raw → Lowercase → Hapus URL saja

In [ ]:
print("\n" + "=" * 65)

# ============================================================
# JUDUL ANALISIS VOCABULARY PIPELINE MINIMAL
# ============================================================

# Pipeline minimal merupakan preprocessing ringan
# yang hanya melakukan:
# 1. Lowercase
# 2. Penghapusan URL

# Tujuan analisis:
# 1. Membandingkan efek preprocessing ringan
#    dengan preprocessing maksimal
# 2. Mengukur seberapa besar reduksi vocabulary
# 3. Melihat perbedaan kompleksitas fitur
# 4. Menjadi bahan analisis eksperimen model

print("  MENGHITUNG VOCABULARY — PIPELINE MINIMAL")

print("=" * 65)

# ============================================================
# DICTIONARY UNTUK MENYIMPAN HASIL
# ============================================================

# vocab_min digunakan untuk menyimpan:
# key   -> nama tahapan preprocessing
# value -> jumlah vocabulary (kata unik)

vocab_min = {}

# ============================================================
# TAHAP 1 — RAW TEXT (DATA ASLI)
# ============================================================

# Menghitung jumlah kata unik sebelum preprocessing

# ' '.join(s1)
# menggabungkan seluruh dokumen menjadi satu string

# split()
# memecah teks menjadi token/kata

# set()
# mengambil kata unik saja

# len()
# menghitung jumlah kata unik

vocab_min['1. Raw\n(Asli)'] = len(
    set(
        ' '.join(s1).split()
    )
)

# ============================================================
# TAHAP 2 — LOWERCASE
# ============================================================

# Mengubah seluruh huruf menjadi kecil

# Contoh:
# "Presiden" -> "presiden"
# "COVID"    -> "covid"

# Tujuan:
# mengurangi perbedaan kata akibat kapitalisasi

s_min2 = s1.apply(tahap_lowercase)

# Menghitung vocabulary setelah lowercase
vocab_min['2. Lowercase'] = len(
    set(
        ' '.join(s_min2).split()
    )
)

# ============================================================
# TAHAP 3 — HAPUS URL
# ============================================================

# tahap_hapus_url hanya menghapus link/URL
# tanpa cleaning lain seperti:
# - stopword
# - stemming
# - simbol
# - angka

# Contoh URL yang dihapus:
# https://...
# www....
# bit.ly/...

# Tujuan:
# menghilangkan noise sederhana dari media sosial

s_min3 = s_min2.apply(tahap_hapus_url)

# Menghitung vocabulary setelah penghapusan URL
vocab_min['3. Hapus URL\n(Output Akhir)'] = len(
    set(
        ' '.join(s_min3).split()
    )
)

# ============================================================
# MENAMPILKAN HEADER TABEL
# ============================================================

# <35 -> rata kiri lebar 35 karakter
# >10 -> rata kanan lebar 10 karakter

print(f"\n{'Tahap':<35} {'Kata Unik':>10} {'Reduksi':>10}")

# Garis pemisah tabel
print("-" * 57)

# ============================================================
# MENAMPILKAN HASIL TIAP TAHAP
# ============================================================

for t, v in vocab_min.items():

    # ========================================================
    # MENGHITUNG PERSENTASE REDUKSI
    # ========================================================

    # Rumus:
    # (1 - jumlah_vocab_saat_ini / vocab_awal) * 100

    # Tujuan:
    # melihat seberapa besar preprocessing
    # berhasil mengurangi jumlah kata unik

    print(

        # Nama tahap preprocessing
        f"{t.replace(chr(10),' '):<35}"

        # Jumlah vocabulary
        f"{v:>10,}"

        # Persentase reduksi
        f"{(1-v/vocab_raw)*100:>9.1f}%"
    )

# ============================================================
# MENYIMPAN NILAI UNTUK ANALISIS PERBANDINGAN
# ============================================================

# ============================================================
# JUMLAH VOCABULARY DATA ASLI
# ============================================================

# Digunakan sebagai baseline/perbandingan utama

val_raw = vocab_maks['1. Raw\n(Asli)']

# ============================================================
# JUMLAH VOCABULARY PIPELINE MINIMAL
# ============================================================

# Vocabulary akhir setelah:
# lowercase + hapus URL

val_minimal = vocab_min['3. Hapus URL\n(Output Akhir)']

# ============================================================
# JUMLAH VOCABULARY PIPELINE MAKSIMAL
# ============================================================

# Vocabulary akhir setelah:
# lowercase + cleaning + stopword + stemming

val_maksimal = vocab_maks['5. Stemming\n(Output Akhir)']

# ============================================================
# TUJUAN PENYIMPANAN VARIABEL
# ============================================================

# Ketiga variabel ini biasanya digunakan untuk:
# 1. Visualisasi perbandingan vocabulary
# 2. Analisis reduksi fitur
# 3. Perbandingan preprocessing minimal vs maksimal
# 4. Analisis dampak preprocessing terhadap model ML


  MENGHITUNG VOCABULARY — PIPELINE MINIMAL

Tahap                                Kata Unik    Reduksi
---------------------------------------------------------
1. Raw (Asli)                            5,080       0.0%
2. Lowercase                             4,383      13.7%
3. Hapus URL (Output Akhir)              4,209      17.1%


### 5.4 Dampak per tahap PIPELINE MAKSIMAL

In [ ]:
# ============================================================
# MENYIAPKAN DATA VISUALISASI
# ============================================================

# labels_m
# berisi nama tahapan preprocessing

# Contoh:
# ['Raw', 'Lowercase', 'Cleaning', ...]

labels_m = list(vocab_maks.keys())

# ============================================================
# NILAI JUMLAH VOCABULARY
# ============================================================

# values_m
# berisi jumlah kata unik pada tiap tahap preprocessing

values_m = list(vocab_maks.values())

# ============================================================
# WARNA VISUALISASI
# ============================================================

# Setiap tahap preprocessing diberikan warna berbeda
# agar lebih mudah dibedakan secara visual

palette = [

    # Raw text
    '#2C3E50',

    # Lowercase
    '#2980B9',

    # Cleaning
    '#F39C12',

    # Stopword removal
    '#8E44AD',

    # Stemming
    '#E74C3C'
]

# ============================================================
# MEMBUAT AREA VISUALISASI
# ============================================================

# Membuat figure dengan 2 subplot:
# kiri  -> Bar Chart
# kanan -> Line Chart

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ============================================================
# JUDUL UTAMA VISUALISASI
# ============================================================

# Visualisasi ini bertujuan untuk:
# 1. Melihat perubahan ukuran vocabulary
# 2. Mengukur dampak preprocessing
# 3. Menunjukkan reduksi fitur teks
# 4. Membandingkan tiap tahap preprocessing

fig.suptitle(

    'Dampak Setiap Tahap Preprocessing terhadap Ukuran Kosakata\n'

    'Skenario 2: Pembersihan Maksimal (bersihkan_teks_total)',

    fontsize=11,

    fontweight='bold'
)

# ============================================================
# SUBPLOT 1 — BAR CHART
# ============================================================

# Bar chart digunakan untuk:
# 1. Membandingkan jumlah vocabulary tiap tahap
# 2. Melihat penurunan vocabulary secara visual
# 3. Menunjukkan tahap preprocessing paling berpengaruh

bars = axes[0].bar(

    # Posisi batang
    range(len(labels_m)),

    # Nilai vocabulary
    values_m,

    # Warna batang
    color=palette,

    # Warna garis tepi
    edgecolor='white',

    # Transparansi warna
    alpha=0.9,

    # Lebar batang
    width=0.6
)

# ============================================================
# PENGATURAN LABEL X
# ============================================================

# Menentukan posisi label sumbu X
axes[0].set_xticks(range(len(labels_m)))

# Menampilkan nama tahap preprocessing
axes[0].set_xticklabels(labels_m, fontsize=8)

# ============================================================
# JUDUL DAN LABEL BAR CHART
# ============================================================

axes[0].set_title(
    'Ukuran Kosakata per Tahap',
    fontweight='bold'
)

axes[0].set_ylabel('Jumlah Kata Unik')

# ============================================================
# MENAMPILKAN ANGKA DI ATAS BATANG
# ============================================================

for bar, val in zip(bars, values_m):

    axes[0].text(

        # Posisi horizontal teks
        bar.get_x() + bar.get_width()/2,

        # Posisi vertikal teks
        bar.get_height() + max(values_m)*0.015,

        # Isi teks
        f'{val:,}',

        # Posisi horizontal teks
        ha='center',

        # Posisi vertikal teks
        va='bottom',

        # Ukuran font
        fontsize=9,

        # Ketebalan font
        fontweight='bold'
    )

# ============================================================
# SUBPLOT 2 — LINE CHART
# ============================================================

# Line chart digunakan untuk:
# 1. Menunjukkan tren penurunan vocabulary
# 2. Memperlihatkan pola reduksi kata
# 3. Memudahkan interpretasi perubahan bertahap

axes[1].plot(

    # Posisi titik X
    range(len(labels_m)),

    # Nilai vocabulary
    values_m,

    # Bentuk marker titik
    marker='o',

    # Warna garis
    color='#E74C3C',

    # Ketebalan garis
    linewidth=2.5,

    # Ukuran marker
    markersize=10,

    # Warna isi marker
    markerfacecolor='white',

    # Ketebalan garis marker
    markeredgewidth=2.5
)

# ============================================================
# MEMBERIKAN AREA FILL PADA GRAFIK
# ============================================================

# fill_between()
# digunakan agar tren penurunan lebih jelas

axes[1].fill_between(

    # Posisi X
    range(len(labels_m)),

    # Nilai Y
    values_m,

    # Transparansi area
    alpha=0.12,

    # Warna area
    color='#E74C3C'
)

# ============================================================
# PENGATURAN LABEL X LINE CHART
# ============================================================

axes[1].set_xticks(range(len(labels_m)))

axes[1].set_xticklabels(labels_m, fontsize=8)

# ============================================================
# JUDUL DAN LABEL LINE CHART
# ============================================================

axes[1].set_title(
    'Tren Penurunan Ukuran Kosakata',
    fontweight='bold'
)

axes[1].set_ylabel('Jumlah Kata Unik')

# ============================================================
# MENAMPILKAN LABEL ANGKA PADA SETIAP TITIK
# ============================================================

for x, y in zip(range(len(labels_m)), values_m):

    axes[1].annotate(

        # Isi teks
        f'{y:,}',

        # Posisi titik
        (x, y),

        # Posisi teks relatif terhadap titik
        textcoords="offset points",

        # Offset posisi teks
        xytext=(0, 12),

        # Posisi horizontal teks
        ha='center',

        # Ukuran font
        fontsize=9,

        # Ketebalan font
        fontweight='bold'
    )

# ============================================================
# MENGATUR TATA LETAK
# ============================================================

# Agar elemen visual tidak bertabrakan
plt.tight_layout()

# ============================================================
# MENYIMPAN VISUALISASI
# ============================================================

# Menyimpan hasil visualisasi ke folder gambar

plt.savefig(

    '../gambar/gambar_5A_dampak_per_tahap_maksimal.png',

    # Resolusi gambar
    dpi=150,

    # Menghindari gambar terpotong
    bbox_inches='tight'
)

# ============================================================
# MENUTUP FIGURE
# ============================================================

# Menghemat penggunaan memori
plt.close()

# ============================================================
# INFORMASI PENYIMPANAN FILE
# ============================================================

print("\n[SAVE] gambar_5A_dampak_per_tahap_maksimal.png")


[SAVE] gambar_5A_dampak_per_tahap_maksimal.png


### 5.5 Visualisasi Perbandingan HASIL AKHIR Minimal vs Maksimal

In [ ]:
# ============================================================
# MEMBUAT AREA VISUALISASI
# ============================================================

# Membuat figure dengan 2 subplot:
# kiri  -> Bar chart ukuran vocabulary
# kanan -> Horizontal bar chart persentase reduksi

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ============================================================
# JUDUL UTAMA VISUALISASI
# ============================================================

# Visualisasi ini bertujuan untuk:
# 1. Membandingkan preprocessing minimal vs maksimal
# 2. Melihat dampak preprocessing terhadap vocabulary
# 3. Mengukur tingkat reduksi fitur teks
# 4. Menentukan preprocessing paling agresif

fig.suptitle(

    'Perbandingan Hasil Akhir: '
    'Skenario 1 (Minimal) vs Skenario 2 (Maksimal)',

    fontsize=12,

    fontweight='bold'
)

# ============================================================
# SUBPLOT 1 — BAR CHART UKURAN VOCABULARY
# ============================================================

# ============================================================
# LABEL KATEGORI
# ============================================================

kategori = [

    # Data asli tanpa preprocessing
    'Raw\n(Asli)',

    # Pipeline preprocessing ringan
    'Skenario 1\n(bersihkan_teks_minimal)',

    # Pipeline preprocessing lengkap
    'Skenario 2\n(bersihkan_teks_total)'
]

# ============================================================
# NILAI VOCABULARY
# ============================================================

# val_raw      -> vocabulary data asli
# val_minimal  -> vocabulary preprocessing minimal
# val_maksimal -> vocabulary preprocessing maksimal

values_perb = [

    val_raw,

    val_minimal,

    val_maksimal
]

# ============================================================
# WARNA BATANG
# ============================================================

colors_perb = [

    # Raw text
    '#2C3E50',

    # Minimal preprocessing
    '#F39C12',

    # Maksimal preprocessing
    '#E74C3C'
]

# ============================================================
# MEMBUAT BAR CHART
# ============================================================

bars = axes[0].bar(

    # Posisi batang
    range(3),

    # Nilai vocabulary
    values_perb,

    # Warna batang
    color=colors_perb,

    # Warna garis tepi
    edgecolor='white',

    # Transparansi warna
    alpha=0.9,

    # Lebar batang
    width=0.5
)

# ============================================================
# PENGATURAN LABEL X
# ============================================================

axes[0].set_xticks(range(3))

axes[0].set_xticklabels(
    kategori,
    fontsize=9
)

# ============================================================
# JUDUL DAN LABEL BAR CHART
# ============================================================

axes[0].set_title(
    'Ukuran Kosakata Hasil Akhir per Skenario',
    fontweight='bold'
)

axes[0].set_ylabel('Jumlah Kata Unik')

# ============================================================
# MENGATUR BATAS SUMBU Y
# ============================================================

# Memberikan ruang tambahan
# agar label angka tidak terpotong

axes[0].set_ylim(

    0,

    max(values_perb) * 1.22
)

# ============================================================
# MENAMPILKAN LABEL DI ATAS BATANG
# ============================================================

for bar, val in zip(bars, values_perb):

    # ========================================================
    # MENGHITUNG PERSENTASE REDUKSI
    # ========================================================

    # Jika raw -> reduksi = 0
    # Jika preprocessing -> hitung reduksi terhadap raw

    reduksi = (

        (1 - val / val_raw) * 100

        if val != val_raw

        else 0
    )

    # ========================================================
    # MEMBUAT LABEL TEKS
    # ========================================================

    # baseline -> data asli
    # reduksi  -> hasil preprocessing

    label = (

        f'{val:,}\n(baseline)'

        if reduksi == 0

        else f'{val:,}\n({reduksi:.1f}% reduksi)'
    )

    # ========================================================
    # MENAMPILKAN TEKS
    # ========================================================

    axes[0].text(

        # Posisi horizontal teks
        bar.get_x() + bar.get_width()/2,

        # Posisi vertikal teks
        bar.get_height() + max(values_perb)*0.01,

        # Isi label
        label,

        # Posisi horizontal teks
        ha='center',

        # Posisi vertikal teks
        va='bottom',

        # Ukuran font
        fontsize=9,

        # Ketebalan font
        fontweight='bold'
    )

# ============================================================
# SUBPLOT 2 — HORIZONTAL BAR CHART REDUKSI
# ============================================================

# ============================================================
# LABEL REDUKSI
# ============================================================

label_r = [

    # Baseline data asli
    'Raw\n(Baseline)',

    # Pipeline minimal
    'Skenario 1\n(Minimal)',

    # Pipeline maksimal
    'Skenario 2\n(Maksimal)'
]

# ============================================================
# NILAI PERSENTASE REDUKSI
# ============================================================

# Rumus:
# (1 - vocab_setelah / vocab_awal) * 100

reduksi_r = [

    # Raw tidak memiliki reduksi
    0,

    # Reduksi preprocessing minimal
    (1 - val_minimal / val_raw) * 100,

    # Reduksi preprocessing maksimal
    (1 - val_maksimal / val_raw) * 100
]

# ============================================================
# WARNA HORIZONTAL BAR
# ============================================================

colors_r = [

    # Baseline
    '#95A5A6',

    # Minimal
    '#F39C12',

    # Maksimal
    '#E74C3C'
]

# ============================================================
# MEMBUAT HORIZONTAL BAR CHART
# ============================================================

hbars = axes[1].barh(

    # Posisi batang horizontal
    range(3),

    # Nilai reduksi
    reduksi_r,

    # Warna batang
    color=colors_r,

    # Warna garis tepi
    edgecolor='white',

    # Transparansi
    alpha=0.9,

    # Tinggi batang
    height=0.5
)

# ============================================================
# PENGATURAN LABEL Y
# ============================================================

axes[1].set_yticks(range(3))

axes[1].set_yticklabels(
    label_r,
    fontsize=9
)

# ============================================================
# JUDUL DAN LABEL HORIZONTAL BAR
# ============================================================

axes[1].set_title(
    'Persentase Reduksi Kosakata dari Raw (%)',
    fontweight='bold'
)

axes[1].set_xlabel('Reduksi (%)')

# ============================================================
# MENGATUR BATAS SUMBU X
# ============================================================

# Maksimum 110 agar ada ruang label teks

axes[1].set_xlim(0, 110)

# ============================================================
# MENAMPILKAN LABEL PERSENTASE
# ============================================================

for hbar, val in zip(hbars, reduksi_r):

    axes[1].text(

        # Posisi horizontal teks
        val + 1,

        # Posisi vertikal teks
        hbar.get_y() + hbar.get_height()/2,

        # Isi teks
        f'{val:.1f}%',

        # Posisi vertikal teks
        va='center',

        # Ukuran font
        fontsize=10,

        # Ketebalan font
        fontweight='bold'
    )

# ============================================================
# MENGATUR TATA LETAK
# ============================================================

# Agar elemen visual tidak saling bertabrakan

plt.tight_layout()

# ============================================================
# MENYIMPAN VISUALISASI
# ============================================================

plt.savefig(

    '../gambar/gambar_5B_perbandingan_minimal_vs_maksimal.png',

    # Resolusi gambar
    dpi=150,

    # Menghindari gambar terpotong
    bbox_inches='tight'
)

# ============================================================
# MENUTUP FIGURE
# ============================================================

# Menghemat penggunaan memori

plt.close()

# ============================================================
# INFORMASI PENYIMPANAN FILE
# ============================================================

print("[SAVE] gambar_5B_perbandingan_minimal_vs_maksimal.png")

[SAVE] gambar_5B_perbandingan_minimal_vs_maksimal.png


### 5.6 Melakukan Simulasi Demo Alur Pembersihan

In [31]:
print("\n[INFO] Menyiapkan data demo untuk tabel alur...")
teks_demo = "WASPADA!! Vaksin COVID-19 mengandung Microchip 5G yang BERBAHAYA! Cek http://hoax.com 😡 @user123 #HoaxAlert"
 
lc_demo      = tahap_lowercase(teks_demo)
cl_demo      = tahap_cleaning(lc_demo)
sw_demo      = tahap_stopword(cl_demo)
print("   [Stemming untuk tabel demo...]")
st_demo      = tahap_stemming(sw_demo)
lc_min_demo  = tahap_lowercase(teks_demo)
url_min_demo = tahap_hapus_url(lc_min_demo)


[INFO] Menyiapkan data demo untuk tabel alur...
   [Stemming untuk tabel demo...]


#### 5.6.1 Tabel alur PIPELINE MAKSIMAL

In [32]:
fig, ax = plt.subplots(figsize=(16, 7))
ax.axis('off')
data_maks = [
    ['1. Raw (Asli)',                         potong(teks_demo)],
    ['2. Lowercase',                          potong(lc_demo)],
    ['3. Cleaning\n(hapus URL, mention,\nhashtag, non-huruf)', potong(cl_demo)],
    ['4. Stopword Removal',                   potong(sw_demo)],
    ['5. Stemming\n(= Output bersihkan_teks_total)', potong(st_demo)],
]
warna_maks = ['#EBF5FB', '#FDFEFE', '#FEF9E7', '#EAFAF1', '#FDEDEC']
t = ax.table(cellText=data_maks, colLabels=['Tahap', 'Hasil Teks'],
             cellLoc='left', loc='center', colWidths=[0.26, 0.69])
t.auto_set_font_size(False)
t.set_fontsize(8.5)
t.scale(1, 3.2)
for (r, c), cell in t.get_celld().items():
    cell.set_edgecolor('#BDC3C7')
    if r == 0:
        cell.set_facecolor('#1F3864')
        cell.set_text_props(color='white', fontweight='bold')
    else:
        cell.set_facecolor(warna_maks[r - 1])
ax.set_title(
    'Alur Preprocessing Teks — Skenario 2: Pembersihan Maksimal\n'
    '(Pipeline bersihkan_teks_total — digunakan untuk X_train, X_test_clean)',
    fontweight='bold', fontsize=11, pad=20)
plt.tight_layout()
plt.savefig('../gambar/gambar_6A_tabel_alur_maksimal.png', dpi=150, bbox_inches='tight')
plt.close()
print("[SAVE] gambar_6A_tabel_alur_maksimal.png")

[SAVE] gambar_6A_tabel_alur_maksimal.png


#### 5.6.2 Tabel alur PIPELINE MINIMAL

In [33]:
fig, ax = plt.subplots(figsize=(16, 4))
ax.axis('off')
data_min = [
    ['1. Raw (Asli)',                              potong(teks_demo)],
    ['2. Lowercase',                              potong(lc_min_demo)],
    ['3. Hapus URL\n(= Output bersihkan_teks_minimal)', potong(url_min_demo)],
]
warna_min = ['#EBF5FB', '#FDFEFE', '#FEF9E7']
t = ax.table(cellText=data_min, colLabels=['Tahap', 'Hasil Teks'],
             cellLoc='left', loc='center', colWidths=[0.30, 0.65])
t.auto_set_font_size(False)
t.set_fontsize(9)
t.scale(1, 3.0)
for (r, c), cell in t.get_celld().items():
    cell.set_edgecolor('#BDC3C7')
    if r == 0:
        cell.set_facecolor('#784212')
        cell.set_text_props(color='white', fontweight='bold')
    else:
        cell.set_facecolor(warna_min[r - 1])
ax.set_title(
    'Alur Preprocessing Teks — Skenario 1: Pembersihan Minimal\n'
    '(Pipeline bersihkan_teks_minimal — digunakan untuk X_test_noisy)',
    fontweight='bold', fontsize=11, pad=20)
plt.tight_layout()
plt.savefig('../gambar/gambar_6B_tabel_alur_minimal.png', dpi=150, bbox_inches='tight')
plt.close()
print("[SAVE] gambar_6B_tabel_alur_minimal.png")

[SAVE] gambar_6B_tabel_alur_minimal.png


#### 5.6.3 Kedua pipeline BERDAMPINGAN dalam 1 gambar

In [34]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(
    'Perbandingan Alur Preprocessing: Skenario 1 (Minimal) vs Skenario 2 (Maksimal)',
    fontsize=12, fontweight='bold'
)
 
# Panel kiri: Minimal
axes[0].axis('off')
data_kiri = [
    ['1. Raw (Asli)',          potong(teks_demo, 55)],
    ['2. Lowercase',           potong(lc_min_demo, 55)],
    ['3. Hapus URL\n(Output)', potong(url_min_demo, 55)],
]
warna_k = ['#EBF5FB', '#FDFEFE', '#FEF9E7']
t_k = axes[0].table(cellText=data_kiri, colLabels=['Tahap', 'Hasil'],
                    cellLoc='left', loc='center', colWidths=[0.40, 0.55])
t_k.auto_set_font_size(False)
t_k.set_fontsize(8)
t_k.scale(1, 3.5)
for (r, c), cell in t_k.get_celld().items():
    cell.set_edgecolor('#BDC3C7')
    if r == 0:
        cell.set_facecolor('#784212')
        cell.set_text_props(color='white', fontweight='bold')
    else:
        cell.set_facecolor(warna_k[r - 1])
axes[0].set_title('SKENARIO 1 — bersihkan_teks_minimal\n(X_test_noisy: simulasi data kotor)',
                  fontweight='bold', color='#784212', fontsize=10, pad=15)
 
# Panel kanan: Maksimal
axes[1].axis('off')
data_kanan = [
    ['1. Raw (Asli)',                    potong(teks_demo, 55)],
    ['2. Lowercase',                     potong(lc_demo, 55)],
    ['3. Cleaning',                      potong(cl_demo, 55)],
    ['4. Stopword Removal',              potong(sw_demo, 55)],
    ['5. Stemming (Output)',             potong(st_demo, 55)],
]
warna_ka = ['#EBF5FB', '#FDFEFE', '#FEF9E7', '#EAFAF1', '#FDEDEC']
t_ka = axes[1].table(cellText=data_kanan, colLabels=['Tahap', 'Hasil'],
                     cellLoc='left', loc='center', colWidths=[0.35, 0.60])
t_ka.auto_set_font_size(False)
t_ka.set_fontsize(8)
t_ka.scale(1, 2.2)
for (r, c), cell in t_ka.get_celld().items():
    cell.set_edgecolor('#BDC3C7')
    if r == 0:
        cell.set_facecolor('#1F3864')
        cell.set_text_props(color='white', fontweight='bold')
    else:
        cell.set_facecolor(warna_ka[r - 1])
axes[1].set_title('SKENARIO 2 — bersihkan_teks_total\n(X_train, X_test_clean: kondisi ideal)',
                  fontweight='bold', color='#1F3864', fontsize=10, pad=15)
 
plt.tight_layout()
plt.savefig('../gambar/gambar_6C_tabel_alur_berdampingan.png', dpi=150, bbox_inches='tight')
plt.close()
print("[SAVE] gambar_6C_tabel_alur_berdampingan.png")

[SAVE] gambar_6C_tabel_alur_berdampingan.png


### 5.7 Tabel kata sebelum vs sesudah stemming

In [ ]:
print("\n[INFO] Membuat tabel stemming...")

# ============================================================
# DAFTAR CONTOH KATA BERIMBUHAN
# ============================================================

# Daftar ini berisi beberapa kata Bahasa Indonesia
# yang memiliki imbuhan seperti:
# - me-
# - pe-
# - di-
# - ter-
# - ber-
# - per-
# dan variasi turunannya

# Tujuan:
# untuk menunjukkan bagaimana proses stemming
# mengubah kata berimbuhan menjadi kata dasar

contoh_kata = [

    # Variasi kata "sebar"
    "menyebarkan",
    "penyebaran",
    "disebarkan",

    # Variasi kata "beri"
    "memberikan",
    "pemberian",
    "diberikan",

    # Variasi kata "bukti"
    "membuktikan",
    "pembuktian",
    "terbukti",

    # Variasi kata "bahaya"
    "berbahaya",
    "membahayakan",

    # Variasi kata "sesat"
    "menyesatkan",
    "penyesatan",

    # Variasi kata "berita"
    "memberitakan",
    "pemberitaan",

    # Variasi kata "klarifikasi"
    "mengklarifikasi",
]

# ============================================================
# PROSES STEMMING
# ============================================================

# stemmer.stem(k)
# digunakan untuk mengubah kata berimbuhan
# menjadi bentuk dasar/root word

# Contoh:
# "menyebarkan" -> "sebar"
# "pemberitaan" -> "berita"

# Hasil disimpan dalam bentuk tuple:
# (kata_asli, kata_hasil_stemming)

hasil_stemming = [

    (k, stemmer.stem(k))

    for k in contoh_kata
]

# ============================================================
# MENAMPILKAN HEADER TABEL DI TERMINAL
# ============================================================

print(
    f"\n{'No':<5} "
    f"{'Sebelum Stemming':<25} "
    f"{'Sesudah Stemming':<20}"
)

# Garis pemisah tabel
print("-" * 52)

# ============================================================
# MENAMPILKAN HASIL STEMMING
# ============================================================

# enumerate(..., 1)
# digunakan agar nomor dimulai dari 1

for i, (s, h) in enumerate(hasil_stemming, 1):

    # s -> sebelum stemming
    # h -> hasil stemming

    print(

        f"{i:<5} "

        f"{s:<25} "

        f"{h:<20}"
    )

# ============================================================
# MEMBUAT VISUALISASI TABEL STEMMING
# ============================================================

# Figure digunakan untuk membuat tabel visual
# yang nantinya dapat dimasukkan ke laporan skripsi

fig, ax = plt.subplots(figsize=(10, 8))

# ============================================================
# MENONAKTIFKAN AXIS
# ============================================================

# Karena hanya ingin menampilkan tabel
# tanpa sumbu X dan Y

ax.axis('off')

# ============================================================
# MENYIAPKAN DATA TABEL
# ============================================================

# Format data:
# [nomor, kata_sebelum, kata_sesudah]

data_stem = [

    [str(i), s, h]

    for i, (s, h)

    in enumerate(hasil_stemming, 1)
]

# ============================================================
# MEMBUAT TABEL
# ============================================================

t_stem = ax.table(

    # Isi data tabel
    cellText=data_stem,

    # Header kolom
    colLabels=[

        'No',

        'Kata Berimbuhan (Sebelum Stemming)',

        'Kata Dasar (Sesudah Stemming)'
    ],

    # Posisi teks di dalam cell
    cellLoc='left',

    # Posisi tabel
    loc='center',

    # Lebar tiap kolom
    colWidths=[0.07, 0.50, 0.43]
)

# ============================================================
# PENGATURAN UKURAN FONT
# ============================================================

# Menonaktifkan autosize font
t_stem.auto_set_font_size(False)

# Menentukan ukuran font manual
t_stem.set_fontsize(10)

# ============================================================
# MENGATUR SKALA TABEL
# ============================================================

# scale(x, y)
# x -> lebar tabel
# y -> tinggi tabel

t_stem.scale(1, 1.9)

# ============================================================
# STYLING TABEL
# ============================================================

# get_celld()
# mengambil seluruh cell pada tabel

for (r, c), cell in t_stem.get_celld().items():

    # ========================================================
    # MEMBERIKAN WARNA GARIS TABEL
    # ========================================================

    cell.set_edgecolor('#BDC3C7')

    # ========================================================
    # STYLING HEADER
    # ========================================================

    if r == 0:

        # Warna background header
        cell.set_facecolor('#1F3864')

        # Warna dan ketebalan teks header
        cell.set_text_props(
            color='white',
            fontweight='bold'
        )

    # ========================================================
    # STYLING BARIS GANJIL
    # ========================================================

    elif r % 2 == 1:

        # Warna biru muda
        cell.set_facecolor('#EBF5FB')

    # ========================================================
    # STYLING BARIS GENAP
    # ========================================================

    else:

        # Warna putih
        cell.set_facecolor('#FDFEFE')

# ============================================================
# MENAMBAHKAN JUDUL TABEL
# ============================================================

# Judul menjelaskan:
# 1. Algoritma stemming yang digunakan
# 2. Implementasi menggunakan library Sastrawi

ax.set_title(

    'Contoh Hasil Stemming Algoritma Nazief & Adriani\n'

    '(Implementasi via Pustaka Sastrawi)',

    fontweight='bold',

    fontsize=11,

    pad=20
)

# ============================================================
# MENGATUR TATA LETAK
# ============================================================

# Agar tabel tidak terpotong
plt.tight_layout()

# ============================================================
# MENYIMPAN HASIL TABEL
# ============================================================

plt.savefig(

    '../gambar/gambar_6D_tabel_stemming.png',

    # Resolusi gambar
    dpi=150,

    # Menghindari gambar terpotong
    bbox_inches='tight'
)

# ============================================================
# MENUTUP FIGURE
# ============================================================

# Menghemat penggunaan memori
plt.close()

# ============================================================
# INFORMASI PENYIMPANAN FILE
# ============================================================

print("[SAVE] gambar_6D_tabel_stemming.png")


[INFO] Membuat tabel stemming...

No    Sebelum Stemming          Sesudah Stemming    
----------------------------------------------------
1     menyebarkan               sebar               
2     penyebaran                sebar               
3     disebarkan                sebar               
4     memberikan                beri                
5     pemberian                 beri                
6     diberikan                 beri                
7     membuktikan               bukti               
8     pembuktian                bukti               
9     terbukti                  bukti               
10    berbahaya                 bahaya              
11    membahayakan              bahaya              
12    menyesatkan               sesat               
13    penyesatan                sesat               
14    memberitakan              berita              
15    pemberitaan               berita              
16    mengklarifikasi           klarifikasi         
[SAVE] gamb